# ITCD Domain-Aware on Google Colab

Notebook ini menjalankan source code dari repository GitHub. Simpan input GeoTIFF dan model weights di Google Drive, ubah nilai konfigurasi pada cell berikut, lalu jalankan cell secara berurutan.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone source code dari GitHub

In [ ]:
!git clone --depth 1 https://github.com/ambarwariagus/itcd-domain-aware.git /content/itcd-domain-aware
%cd /content/itcd-domain-aware

## 3. Install dependency Colab

Notebook ini mengikuti dependency yang sudah terbukti berjalan: `segment-geospatial` dan Detectron2 dari source.

In [ ]:
!pip install -q segment-geospatial
!python -m pip install -q pyyaml==5.1
!git clone --depth 1 https://github.com/facebookresearch/detectron2.git /content/detectron2
import os
import sys
import distutils.core

dist = distutils.core.run_setup('/content/detectron2/setup.py')
!python -m pip install -q {' '.join([f"'{requirement}'" for requirement in dist.install_requires])}
sys.path.insert(0, '/content/detectron2')
%cd /content/itcd-domain-aware

## 4. Konfigurasi Google Drive

Ubah tiga path berikut sesuai lokasi file Anda di Google Drive. Struktur model harus berisi `config.yaml` dan `model_best_AP50.pth`.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ITCD_Domain_Aware')
INPUT_IMAGE = DRIVE_ROOT / 'input' / 'your_image.tif'
MODEL_DIR = DRIVE_ROOT / 'models' / '250525_R50_RESTOR_fold_0'
OUTPUT_DIR = DRIVE_ROOT / 'output'

assert INPUT_IMAGE.exists(), f'Input tidak ditemukan: {INPUT_IMAGE}'
assert (MODEL_DIR / 'config.yaml').exists(), f'Model config tidak ditemukan: {MODEL_DIR / "config.yaml"}'
assert (MODEL_DIR / 'model_best_AP50.pth').exists(), f'Model weights tidak ditemukan: {MODEL_DIR / "model_best_AP50.pth"}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Input: {INPUT_IMAGE}')
print(f'Model: {MODEL_DIR}')
print(f'Output: {OUTPUT_DIR}')

## 5. Buat konfigurasi pipeline

Ubah `nms_algorithm` menjadi `dc_gnms` untuk memakai Divide-and-Conquer Geometric NMS.

In [ ]:
import yaml

config = {
    'input': {'image_path': str(INPUT_IMAGE)},
    'tiling': {
        'tile_size': [1024, 1024],
        'overlap': 256,
        'output_dir': str(OUTPUT_DIR / 'tiles'),
    },
    'model': {
        'config_path': str(MODEL_DIR / 'config.yaml'),
        'weights_path': str(MODEL_DIR / 'model_best_AP50.pth'),
        'score_threshold': 0.5,
        'nms_threshold': 0.3,
    },
    'postprocessing': {
        'output_geojson_dir': str(OUTPUT_DIR / 'geojson'),
        'nms_algorithm': 'spatial_nms',
        'spatial_nms': {
            'iou_threshold': 0.5,
            'confidence_threshold': 0.5,
            'area_threshold': 2.0,
            'edge_shift': 1.0,
        },
        'dc_gnms': {
            'box_filter_shift': 1.0,
            'overlap_threshold': 0.5,
            'cross_zone_threshold': 0.15,
            'simplify_tolerance': 0.05,
        },
    },
    'output': {
        'final_output_dir': str(OUTPUT_DIR / 'final'),
        'output_filename': 'tree_crowns.gpkg',
        'keep_intermediate': True,
    },
    'logging': {
        'level': 'INFO',
        'save_log': True,
        'log_file': str(OUTPUT_DIR / 'pipeline.log'),
    },
}

config_path = '/content/itcd-domain-aware/config_colab.yaml'
with open(config_path, 'w') as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(config_path)

## 6. Jalankan seluruh pipeline

Output akhir disimpan di `OUTPUT_DIR/final/tree_crowns.gpkg`. Untuk menjalankan tahap tertentu, tambahkan `--step tiling`, `--step inference`, `--step conversion`, atau `--step nms`.

In [ ]:
!python main.py --config /content/itcd-domain-aware/config_colab.yaml

## 7. Lokasi hasil

In [ ]:
final_output = OUTPUT_DIR / 'final' / 'tree_crowns.gpkg'
print(final_output)
print(f'Tersedia: {final_output.exists()}')